# Chapter 4 Upgrade: Distributed Circuit Breaker + Metrics Simulation

This notebook extends the Chapter 4 deployment notebook with a more production-oriented design:

- a **distributed-style circuit breaker state store**
- **retry with exponential backoff**
- **metrics collection**
- **dashboard-style summaries**
- **graceful fallback behavior**
- a simple **reset / half-open recovery flow**

It is still fully self-contained and runnable locally.

## Why this matters
Chapter 4 emphasizes resilience patterns such as circuit breakers, bulkheads, timeout/retry wrappers, and failover behavior for high-throughput agent systems. It also recommends observability and fine-grained runtime monitoring for latency, cost, and failures. This notebook operationalizes those ideas in a compact simulation.

## Design goal

In the chapter's code, the circuit breaker uses module-level globals and notes that a shared store such as Redis should be used in distributed deployments.

This notebook simulates that production direction by introducing:

- a shared breaker state object
- a metrics registry
- per-endpoint tracking
- dashboard-ready summaries
- a half-open recovery mode

That gives you a closer approximation of how a real microservice deployment behaves.

In [ ]:
# Optional if needed:
# !pip install tenacity pandas matplotlib

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional
from tenacity import retry, stop_after_attempt, wait_exponential, RetryError
import random
import time
import math

import pandas as pd
import matplotlib.pyplot as plt

## 1) Simulated shared breaker store

In [ ]:
@dataclass
class BreakerState:
    state: str = "CLOSED"          # CLOSED, OPEN, HALF_OPEN
    failure_count: int = 0
    failure_threshold: int = 3
    opened_at: Optional[float] = None
    reset_timeout_sec: float = 5.0

class SharedBreakerStore:
    """
    A lightweight in-memory stand-in for Redis or another shared store.
    In production, this would be backed by Redis, DynamoDB, etc.
    """
    def __init__(self):
        self._store: Dict[str, BreakerState] = {}

    def get(self, key: str) -> BreakerState:
        if key not in self._store:
            self._store[key] = BreakerState()
        return self._store[key]

    def set(self, key: str, state: BreakerState):
        self._store[key] = state

    def snapshot(self) -> Dict[str, Dict[str, Any]]:
        out = {}
        for k, v in self._store.items():
            out[k] = {
                "state": v.state,
                "failure_count": v.failure_count,
                "failure_threshold": v.failure_threshold,
                "opened_at": v.opened_at,
                "reset_timeout_sec": v.reset_timeout_sec,
            }
        return out

breaker_store = SharedBreakerStore()

## 2) Metrics registry

In [ ]:
@dataclass
class MetricsRegistry:
    events: List[Dict[str, Any]] = field(default_factory=list)

    def log(self, endpoint: str, event_type: str, latency_ms: float, result: str, fallback: bool, detail: str = ""):
        self.events.append({
            "timestamp": time.time(),
            "endpoint": endpoint,
            "event_type": event_type,
            "latency_ms": latency_ms,
            "result": result,
            "fallback": fallback,
            "detail": detail
        })

    def dataframe(self) -> pd.DataFrame:
        if not self.events:
            return pd.DataFrame(columns=["timestamp", "endpoint", "event_type", "latency_ms", "result", "fallback", "detail"])
        return pd.DataFrame(self.events)

metrics = MetricsRegistry()

## 3) Simulated external tool API

In [ ]:
ENDPOINT_BEHAVIOR = {
    "search_api": {"failure_rate": 0.20, "base_latency_ms": 120},
    "payments_api": {"failure_rate": 0.55, "base_latency_ms": 220},
    "profile_api": {"failure_rate": 0.10, "base_latency_ms": 80},
}

def simulated_tool_api(endpoint: str, payload: dict) -> dict:
    if endpoint not in ENDPOINT_BEHAVIOR:
        raise ValueError(f"Unknown endpoint: {endpoint}")

    cfg = ENDPOINT_BEHAVIOR[endpoint]
    simulated_latency = max(10, random.gauss(cfg["base_latency_ms"], cfg["base_latency_ms"] * 0.25))
    time.sleep(simulated_latency / 1000.0)

    if random.random() < cfg["failure_rate"]:
        raise RuntimeError(f"Simulated failure for {endpoint}")

    return {
        "status": "ok",
        "endpoint": endpoint,
        "payload": payload,
        "data": f"Processed request for {payload}"
    }

## 4) Retry wrapper

In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=4))
def call_tool_with_retry(endpoint: str, payload: dict) -> dict:
    return simulated_tool_api(endpoint, payload)

## 5) Distributed-style circuit breaker wrapper

In [ ]:
def maybe_transition_open_to_half_open(breaker: BreakerState):
    if breaker.state == "OPEN" and breaker.opened_at is not None:
        if (time.time() - breaker.opened_at) >= breaker.reset_timeout_sec:
            breaker.state = "HALF_OPEN"

def protected_tool_call(endpoint: str, payload: dict, store: SharedBreakerStore, metrics: MetricsRegistry) -> dict:
    breaker = store.get(endpoint)
    maybe_transition_open_to_half_open(breaker)

    # Fast fail if still open
    if breaker.state == "OPEN":
        metrics.log(
            endpoint=endpoint,
            event_type="blocked",
            latency_ms=0.0,
            result="blocked_open_circuit",
            fallback=True,
            detail="Circuit open; returning fallback."
        )
        return {"status": "unavailable", "fallback": True, "reason": "open_circuit"}

    start = time.time()
    try:
        result = call_tool_with_retry(endpoint, payload)
        latency_ms = (time.time() - start) * 1000.0

        # Success resets breaker
        breaker.failure_count = 0
        breaker.state = "CLOSED"
        breaker.opened_at = None
        store.set(endpoint, breaker)

        metrics.log(
            endpoint=endpoint,
            event_type="success",
            latency_ms=latency_ms,
            result="success",
            fallback=False,
            detail="Tool call succeeded."
        )
        return result

    except RetryError:
        latency_ms = (time.time() - start) * 1000.0
        breaker.failure_count += 1

        # If half-open probe fails, reopen immediately
        if breaker.state == "HALF_OPEN":
            breaker.state = "OPEN"
            breaker.opened_at = time.time()
        elif breaker.failure_count >= breaker.failure_threshold:
            breaker.state = "OPEN"
            breaker.opened_at = time.time()

        store.set(endpoint, breaker)

        metrics.log(
            endpoint=endpoint,
            event_type="failure",
            latency_ms=latency_ms,
            result="fallback_after_failure",
            fallback=True,
            detail=f"Failure count={breaker.failure_count}, breaker_state={breaker.state}"
        )
        return {"status": "unavailable", "fallback": True, "reason": "tool_failure"}

## 6) Single-call demo

In [ ]:
for i in range(5):
    out = protected_tool_call(
        endpoint="payments_api",
        payload={"txn_id": i, "amount": 100 + i},
        store=breaker_store,
        metrics=metrics
    )
    print(f"Call {i+1}: {out}")
    print("Breaker snapshot:", breaker_store.snapshot()["payments_api"])
    print("-" * 80)

## 7) Run a load simulation

In [ ]:
def run_simulation(n_calls_per_endpoint: int = 25, seed: int = 42):
    random.seed(seed)

    endpoints = list(ENDPOINT_BEHAVIOR.keys())
    for endpoint in endpoints:
        for i in range(n_calls_per_endpoint):
            payload = {"request_id": f"{endpoint}-{i}", "value": i}
            _ = protected_tool_call(endpoint, payload, breaker_store, metrics)

run_simulation(n_calls_per_endpoint=20, seed=7)
print("Simulation complete.")

## 8) Inspect raw metrics

In [ ]:
df = metrics.dataframe()
df.head(10)

## 9) Dashboard-style summary

In [ ]:
summary = (
    df.groupby("endpoint")
      .agg(
          total_calls=("endpoint", "count"),
          avg_latency_ms=("latency_ms", "mean"),
          max_latency_ms=("latency_ms", "max"),
          success_count=("result", lambda s: (s == "success").sum()),
          fallback_count=("fallback", "sum"),
          blocked_count=("event_type", lambda s: (s == "blocked").sum()),
          failure_count=("event_type", lambda s: (s == "failure").sum()),
      )
      .reset_index()
)

summary["success_rate"] = summary["success_count"] / summary["total_calls"]
summary["fallback_rate"] = summary["fallback_count"] / summary["total_calls"]
summary["avg_latency_ms"] = summary["avg_latency_ms"].round(2)
summary["max_latency_ms"] = summary["max_latency_ms"].round(2)

summary

## 10) Current breaker state snapshot

In [ ]:
pd.DataFrame.from_dict(breaker_store.snapshot(), orient="index")

## 11) Visualize service behavior

In [ ]:
plot_df = summary.set_index("endpoint")[["success_rate", "fallback_rate"]]
plot_df.plot(kind="bar", figsize=(8, 4), rot=0)
plt.title("Success vs Fallback Rate by Endpoint")
plt.ylabel("Rate")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
lat_df = df[df["latency_ms"] > 0].copy()
lat_df.boxplot(column="latency_ms", by="endpoint", figsize=(8, 4))
plt.title("Latency Distribution by Endpoint")
plt.suptitle("")
plt.ylabel("Latency (ms)")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 12) Observe an open → half-open → closed recovery cycle

In [ ]:
# Force a problematic endpoint into repeated use until it opens
target = "payments_api"

for i in range(10):
    _ = protected_tool_call(target, {"probe": i}, breaker_store, metrics)

print("Before waiting:", breaker_store.snapshot()[target])

# Wait long enough for the breaker to move from OPEN to HALF_OPEN on next call
time.sleep(breaker_store.get(target).reset_timeout_sec + 0.5)

probe_result = protected_tool_call(target, {"probe": "recovery_attempt"}, breaker_store, metrics)

print("Recovery probe result:", probe_result)
print("After recovery attempt:", breaker_store.snapshot()[target])

## 13) What would change in production?

### Shared state
Replace `SharedBreakerStore` with:
- Redis
- DynamoDB
- PostgreSQL
- another low-latency shared state backend

### Telemetry
Send metrics to:
- Prometheus
- Grafana
- OpenTelemetry collector
- Datadog / Cloud Monitoring / Azure Monitor

### Deployment
Run the execution engine as a separate service:
- FastAPI or gRPC
- containerized with Docker
- deployed on Kubernetes
- autoscaled independently from the planner

### Security
Add:
- tool allowlists
- parameter validation
- per-tool auth
- signed service-to-service calls
- audit logs

## 14) Key takeaways

This upgraded notebook demonstrates the production direction implied by Chapter 4:

- **module-level globals are not enough** for distributed deployments
- **circuit breakers need shared state**
- **observability is essential**
- **fallbacks are part of reliability**
- **recovery behavior should be explicit**

This is the difference between a demo tool wrapper and a deployment-oriented execution service.